# Analyse manuelle

Liste des **candidats** : indices présents à la fois dans `indices_pol_all_finetune_wrong.csv` et `indices_medical_all_finetune_wrong.csv`. En bas, tu choisis toi-même l'indice `idx` et une cellule affiche le prompt POL et le prompt MEDICAL associés (depuis `Prompt_cot/cot_test_pol.jsonl` et `cot_test_medical.jsonl`).

In [15]:
import pandas as pd
import json
import os

RESULTS_DIR = "."
PROMPT_COT_DIR = os.path.join(RESULTS_DIR, "Prompt_cot")

df_pol_wrong = pd.read_csv(os.path.join(RESULTS_DIR, "indices_pol_all_finetune_wrong.csv"))
df_medical_wrong = pd.read_csv(os.path.join(RESULTS_DIR, "indices_medical_all_finetune_wrong.csv"))

indices_pol = set(df_pol_wrong["index"].astype(int))
indices_medical = set(df_medical_wrong["index"].astype(int))
candidats = sorted(indices_pol & indices_medical)

print(f"Indices POL (tous finetune wrong): {len(indices_pol)}")
print(f"Indices MEDICAL (tous finetune wrong): {len(indices_medical)}")
print(f"Candidats (présents dans les deux): {len(candidats)}")
print(f"Liste des candidats: {candidats}")

Indices POL (tous finetune wrong): 196
Indices MEDICAL (tous finetune wrong): 174
Candidats (présents dans les deux): 123
Liste des candidats: [1, 16, 29, 30, 47, 78, 94, 135, 139, 144, 145, 155, 165, 182, 209, 231, 241, 246, 267, 280, 282, 302, 318, 336, 367, 388, 392, 393, 410, 412, 423, 428, 447, 455, 457, 475, 493, 497, 508, 514, 576, 582, 583, 587, 592, 600, 601, 602, 660, 731, 745, 748, 800, 801, 811, 820, 821, 830, 834, 835, 866, 872, 891, 896, 906, 909, 922, 951, 966, 976, 980, 992, 996, 1003, 1012, 1014, 1019, 1061, 1073, 1102, 1108, 1115, 1126, 1136, 1157, 1160, 1172, 1187, 1189, 1205, 1210, 1233, 1289, 1296, 1306, 1331, 1342, 1346, 1357, 1363, 1365, 1371, 1382, 1390, 1395, 1420, 1433, 1448, 1461, 1462, 1481, 1485, 1495, 1507, 1516, 1525, 1550, 1556, 1557, 1562, 1565, 1566, 1574]


In [16]:
def load_prompts_by_index(jsonl_path):
    """Charge le JSONL et retourne un dict index -> contenu du message user (prompt)."""
    prompts = {}
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            obj = json.loads(line)
            user_msg = next((m["content"] for m in obj["messages"] if m["role"] == "user"), "")
            prompts[i] = user_msg
    return prompts

path_pol = os.path.join(PROMPT_COT_DIR, "cot_test_pol.jsonl")
path_medical = os.path.join(PROMPT_COT_DIR, "cot_test_medical.jsonl")
prompts_pol = load_prompts_by_index(path_pol)
prompts_medical = load_prompts_by_index(path_medical)

df_eval = pd.read_csv(os.path.join(PROMPT_COT_DIR, "eval_cot_combined_finetune.csv"))
df_eval_baseline = pd.read_csv(os.path.join(PROMPT_COT_DIR, "eval_cot_combined_baseline.csv"))
print(f"Prompts POL chargés: {len(prompts_pol)}, MEDICAL: {len(prompts_medical)}")
print(f"Eval CoT finetune: {len(df_eval)} lignes, baseline: {len(df_eval_baseline)} lignes")

Prompts POL chargés: 1578, MEDICAL: 1578
Eval CoT finetune: 3156 lignes, baseline: 3156 lignes


## Affichage manuel : choisis l'indice `idx` ci-dessous puis exécute la cellule pour afficher les deux prompts, le reasoning et la prediction (depuis l'éval CoT finetune).

In [17]:
# Choisis l'indice (ex. un des candidats) puis exécute la cellule
idx = 29

row_pol = df_eval[(df_eval["index"] == idx) & (df_eval["statement_type"] == "pol")].iloc[0] if len(df_eval[(df_eval["index"] == idx) & (df_eval["statement_type"] == "pol")]) else None
row_medical = df_eval[(df_eval["index"] == idx) & (df_eval["statement_type"] == "medical")].iloc[0] if len(df_eval[(df_eval["index"] == idx) & (df_eval["statement_type"] == "medical")]) else None

print("=" * 80)
print(f"Index = {idx}")
print("=" * 80)
print("\n--- Prompt POL ---\n")
print(prompts_pol.get(idx, "(index absent dans cot_test_pol.jsonl)"))
if row_pol is not None:
    print("\n  [Eval] gold:", row_pol["gold"], " | prediction:", row_pol["prediction"], " | is_correct:", row_pol["is_correct"])
    print("  [Eval] reasoning:\n", row_pol["reasoning"] if pd.notna(row_pol.get("reasoning")) else "(n/a)")
print("\n--- Prompt MEDICAL ---\n")
print(prompts_medical.get(idx, "(index absent dans cot_test_medical.jsonl)"))
if row_medical is not None:
    print("\n  [Eval] gold:", row_medical["gold"], " | prediction:", row_medical["prediction"], " | is_correct:", row_medical["is_correct"])
    print("  [Eval] reasoning:\n", row_medical["reasoning"] if pd.notna(row_medical.get("reasoning")) else "(n/a)")

Index = 29

--- Prompt POL ---

Does the patient with the statement "I had to rush to the ER because of a non-stop nosebleed that had been on for 1 hour. As a 23-year-old, I always had heavy periods, and sometimes, my gums would bleed after going to the dentist or brushing my teeth. I thought that it was not very alarming since my mom also had heavy periods. I did some lab tests that showed that I had increased bleeding time and long clotting time. Otherwise, I'm pretty healthy, and I don't smoke or take any kind of drugs. I drink only at parties from time to time. I have been with my boyfriend for the last 2 years. I did a coagulation study and I got the following results: CB = 0.30 IU/mL and FVIII:C = 0.37 IU/mL." satisfy the following clinical trial admission criteria ?
"Inclusion Criteria:

          -  Women with menorrhagia

          -  18-49 years of age

          -  Regularly occuring menstrual periods

        Exclusion Criteria:

          -  History or presence of clinical

## Même idx — affichage **baseline** (reasoning et prediction)

In [18]:
row_pol_b = df_eval_baseline[(df_eval_baseline["index"] == idx) & (df_eval_baseline["statement_type"] == "pol")].iloc[0] if len(df_eval_baseline[(df_eval_baseline["index"] == idx) & (df_eval_baseline["statement_type"] == "pol")]) else None
row_medical_b = df_eval_baseline[(df_eval_baseline["index"] == idx) & (df_eval_baseline["statement_type"] == "medical")].iloc[0] if len(df_eval_baseline[(df_eval_baseline["index"] == idx) & (df_eval_baseline["statement_type"] == "medical")]) else None

print("=" * 80)
print(f"Index = {idx} — BASELINE")
print("=" * 80)
print("\n--- Prompt POL ---\n")
print(prompts_pol.get(idx, "(index absent dans cot_test_pol.jsonl)"))
if row_pol_b is not None:
    print("\n  [Eval baseline] gold:", row_pol_b["gold"], " | prediction:", row_pol_b["prediction"], " | is_correct:", row_pol_b["is_correct"])
    print("  [Eval baseline] reasoning:\n", row_pol_b["reasoning"] if pd.notna(row_pol_b.get("reasoning")) else "(n/a)")
print("\n--- Prompt MEDICAL ---\n")
print(prompts_medical.get(idx, "(index absent dans cot_test_medical.jsonl)"))
if row_medical_b is not None:
    print("\n  [Eval baseline] gold:", row_medical_b["gold"], " | prediction:", row_medical_b["prediction"], " | is_correct:", row_medical_b["is_correct"])
    print("  [Eval baseline] reasoning:\n", row_medical_b["reasoning"] if pd.notna(row_medical_b.get("reasoning")) else "(n/a)")

Index = 29 — BASELINE

--- Prompt POL ---

Does the patient with the statement "I had to rush to the ER because of a non-stop nosebleed that had been on for 1 hour. As a 23-year-old, I always had heavy periods, and sometimes, my gums would bleed after going to the dentist or brushing my teeth. I thought that it was not very alarming since my mom also had heavy periods. I did some lab tests that showed that I had increased bleeding time and long clotting time. Otherwise, I'm pretty healthy, and I don't smoke or take any kind of drugs. I drink only at parties from time to time. I have been with my boyfriend for the last 2 years. I did a coagulation study and I got the following results: CB = 0.30 IU/mL and FVIII:C = 0.37 IU/mL." satisfy the following clinical trial admission criteria ?
"Inclusion Criteria:

          -  Women with menorrhagia

          -  18-49 years of age

          -  Regularly occuring menstrual periods

        Exclusion Criteria:

          -  History or presence 